# SHAPER — full staged pipeline

**SHAP**ley-guided **PER**turbation learning: cooperative attribution of training-time
augmentation views in contrastive sequential recommendation.

This notebook calls the package/script functions of the repository — it contains **no
scientific logic of its own**. It is restartable and resume-aware: every stage is
idempotent (checkpoint manifest + value-table cache) and refuses confirmatory work
before the pilot-informed archive exists.

The first code cell makes the repository root importable, so the notebook works no
matter where Jupyter's working directory happens to be.

The registered protocol lives in:
- `specs/SHAPER_Implementation_Spec.md` (implementation source of truth)
- `specs/SHAPER_Paper_Structure.md` (paper/interpretation source of truth)

**Safety:** the notebook never launches the full study from one cell. Run the sections
one at a time with `scripts/run_all.py --stage <stage>` or execute cells explicitly.


## 1. Environment

In [ ]:
# Make the repository root importable from ANY working directory
# (Jupyter does not put the repo on sys.path automatically).
import os, pathlib, sys

def _is_repo_root(d: pathlib.Path) -> bool:
    try:
        return (d / 'shaper').is_dir() and (d / 'scripts').is_dir()
    except (PermissionError, OSError):
        return False

def _subdirs_of(d: pathlib.Path):
    try:
        return [p for p in d.iterdir() if p.is_dir()]
    except (PermissionError, OSError):
        return []

REPO_ROOT = None
_candidates = []
try:  # package already importable -> derive the root from its location
    import shaper as _shaper
    _candidates.append(pathlib.Path(_shaper.__file__).resolve().parent.parent)
except Exception:
    pass
_cwd = pathlib.Path.cwd().resolve()
_candidates += [_cwd, *_cwd.parents]                        # cwd and all ancestors
for _p in (_cwd, *_cwd.parents[:2]):                         # + immediate subdirectories
    _candidates += _subdirs_of(_p)
for _cand in _candidates:
    if _is_repo_root(_cand):
        REPO_ROOT = str(_cand)
        break
if REPO_ROOT is None:
    raise RuntimeError('could not locate the SHAPER repository root (a directory containing '
                       'shaper/ and scripts/). Run this notebook from inside the repository.')
for _p in (REPO_ROOT, os.path.join(REPO_ROOT, 'scripts')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print('REPO_ROOT:', REPO_ROOT)

from scripts import run_all
from shaper.config import load_run_config
from shaper.provenance import environment_record
from shaper.schedules import configure_determinism

import json
env = environment_record(configure_determinism(), cwd=REPO_ROOT)
print(json.dumps({k: env[k] for k in ("python", "platform", "cpu_count", "torch_version", "cuda_available", "commit", "dirty")}, indent=2))

## 2. Specification / version

In [ ]:
import shaper
from shaper.provenance import file_hash
print("shaper", shaper.__version__)
print("implementation spec hash:", file_hash(shaper.PROTOCOL_SPEC))
print("paper structure hash:  ", file_hash(shaper.PAPER_SPEC))

## 3. Configuration

In [ ]:
DATASET = "synthetic"   # ml1m | beauty | synthetic
RUN_ID  = "shaper-run"
cfg = load_run_config(DATASET)
print("config hash:", cfg.config_hash())
print("batch size:", cfg.batch_size, "max_len:", cfg.max_len)
print("seeds:", json.dumps(cfg.seeds, indent=2))

## 4. Compute estimate (planning only — the paper reports measured values)

In [ ]:
def estimate_cost():
    from shaper.cost import n_coalition_models_for_scope
    return n_coalition_models_for_scope(cfg).summary()

print(json.dumps(estimate_cost(), indent=2, default=str))

## 5. Data

In [ ]:
run_all.run_stage("data", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 6. Preflight

In [ ]:
from scripts.preflight import run_preflight
print(json.dumps(run_preflight(cfg), indent=2, default=str))

## 7. Recipe calibration (single pass, V_tune only)

In [ ]:
run_all.run_stage("recipe", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False, skip_stress=False,
    stress_only=False))

## 8. Pilots (excluded seeds 1001/1002)

In [ ]:
run_all.run_stage("pilot", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 9. Amendment / archive freeze (no confirmatory model before this)

In [ ]:
run_all.run_stage("amendment", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))
run_all.run_stage("archive", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))
print("archive frozen:", run_all.archive_is_frozen(cfg))

## 10. Game A (exact K=3, all 8 coalitions x 5 confirmatory seeds)

In [ ]:
run_all.run_stage("game-a", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 11. Game B (policy sensitivity, seeds 2001–2003, 6 intermediates only)

In [ ]:
run_all.run_stage("game-b", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 12. NLL/cosine severity diagnostics (frozen rec-only checkpoints)

In [ ]:
# point --rec-only-checkpoints at the recipe empty-coalition checkpoints
print("run: scripts/run_game.py --stage severity --rec-only-checkpoints <paths>")

## 13. K=4 Beauty MC (antithetic permutation, <=10 unique models/seed)

In [ ]:
run_all.run_stage("k4-mc", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 14. Shapley (exact allocation + efficiency residuals + per-user decomposition)

In [ ]:
run_all.run_stage("shapley", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 15. LOO / interactions

In [ ]:
run_all.run_stage("loo", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))
run_all.run_stage("interactions", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 16. RQ3 — behavioural segments (exact Game A only)

In [ ]:
run_all.run_stage("segments", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 17. Weight calibration (alpha path on V_select)

In [ ]:
run_all.run_stage("weight", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 18. Select calibration (locked activation/no-action rule)

In [ ]:
run_all.run_stage("select", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 19. Baseline controls (LOO, gates, direct search, Dirichlet, random)

In [ ]:
run_all.run_stage("controls", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 20. Final intervention test (locked decisions; all eligible users; Tables 7A/7B)

In [ ]:
run_all.run_stage("final-test", cfg, run_all.argparse.Namespace(
    dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
    skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
    synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
    n_permutations=10000, allow_before_archive=False))

## 21. Statistics (seed-level CIs, Holm families, descriptive user analyses)

In [ ]:
from shaper.stats import seed_summary, holm_adjust, paired_seed_effects
print("seed-level paired effects and Holm adjustment are computed by "
      "shaper.stats; confirmatory unit = seeds")

## 22. Tables

In [ ]:
from shaper.report import shapley_table, coalition_value_table
print("tables are rendered to results/runs/<run_id>/tables/ by the stages above")

## 23. Figures

In [ ]:
print("figures are rendered to results/runs/<run_id>/figures/ by the stages above "
      "(Agg backend, headless-safe)")

## 24. Compliance

In [ ]:
print("compliance: scripts/validate_run.py generates spec_compliance.json/.md; "
      "run the test suite with: python scripts/run_all.py --validate")

## 25. Reproducibility

In [ ]:
from shaper.artifacts import RunDirectory
run = RunDirectory(RUN_ID, results_root=cfg.paths["results"]).create(cfg, resume=True)
run.write_reproducibility_report(cfg)
print("reproducibility report:", run.root + "/reproducibility_report.md")

## Notebook helpers (status / validate / estimate-cost / resume)

The helpers below only call package/script functions. The notebook is resume-aware:
every stage is idempotent, so re-running any cell continues from completed work.


In [ ]:
def status():
    import json
    p = os.path.join(cfg.paths["results"], RUN_ID, "manifest.json")
    if os.path.exists(p):
        return json.load(open(p))["stages"]
    return {"note": "run not started"}

def validate():
    import subprocess
    return subprocess.call([sys.executable, "-m", "pytest", "tests", "-q"],
                           cwd=REPO_ROOT)

def estimate_cost():
    from shaper.cost import n_coalition_models_for_scope
    return n_coalition_models_for_scope(cfg).summary()

def resume():
    for stage in run_all.STAGE_ORDER:
        rc = run_all.run_stage(stage, cfg, run_all.argparse.Namespace(
            dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
            skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
            synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
            n_permutations=10000, allow_before_archive=False))
        if rc != 0:
            print("stopped at", stage)
            return rc
    return 0

print(status())